# KARMA – 5-Fold CV Attribution Comparison

Runs `experiments/comparison_realdata_cv.py` on **ETTm1, ETTm2, web_traffic, electricity**.

The test set is split into N consecutive folds. Attribution methods run independently
on each fold. Results are reported as **mean ± std** across folds.

**KARMA** is evaluated once on the full training set; its edge scores are then tested
against each fold's samples.  All other methods re-run per fold.

> **Before running:** Runtime → Change runtime type → **T4 GPU**

## 0 · Configuration — edit this cell

In [ ]:
import os, sys
from pathlib import Path

# ── user-configurable ──────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/KARMA_data'
ROOT         = '/content/KARMA'
DATASETS_RUN = ['etth1', 'etth2', 'ettm1', 'ettm2',
                'exchange_rate', 'beijing_pm25',
                'web_traffic', 'electricity']
N_FOLDS      = 5
TRAIN_EPOCHS = 50
DEVICE       = 'auto'   # 'auto' → cuda if available, else cpu
# ──────────────────────────────────────────────────────────────────────────

ROOT       = Path(ROOT)
DRIVE_ROOT = Path(DRIVE_ROOT)
RAW        = ROOT / 'data' / 'raw'
GEN        = ROOT / 'data' / 'generated'
CKPT_ROOT  = ROOT / 'outputs' / 'checkpoints'
EXP_CKPT   = CKPT_ROOT / 'realdata_cv'
OUT_DIR    = ROOT / 'results' / f'realdata_cv{N_FOLDS}'

DRIVE_RAW  = DRIVE_ROOT / 'raw'
DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
DRIVE_OUT  = DRIVE_ROOT / 'results' / f'realdata_cv{N_FOLDS}'

print('ROOT      :', ROOT)
print('Datasets  :', DATASETS_RUN)
print('N_FOLDS   :', N_FOLDS)

## 1 · Mount Google Drive

Upload raw data files to Drive under `MyDrive/KARMA_data/raw/` before running §4:

```
MyDrive/KARMA_data/
  raw/
    ETTh1.csv
    ETTh2.csv
    ETTm1.csv
    ETTm2.csv
    exchange_rate/
      exchange_rate.txt.gz
    beijing_pm2/
      PRSA_data_2010.1.1-2014.12.31.csv
    electricityloaddiagrams/
      LD2011_2014.txt
    web_traffic/
      kaggle_web_traffic_dataset_without_missing_values.tsf
  checkpoints/
    etth1_lstm/best.pt       etth1_tcn/best.pt
    etth2_lstm/best.pt       etth2_tcn/best.pt
    ettm1_lstm/best.pt       ettm1_tcn/best.pt
    ettm2_lstm/best.pt       ettm2_tcn/best.pt
    exchange_rate_lstm/best.pt  exchange_rate_tcn/best.pt
    beijing_pm25_lstm/best.pt   beijing_pm25_tcn/best.pt
    electricity_lstm/best.pt    electricity_tcn/best.pt
    web_traffic_lstm/best.pt    web_traffic_tcn/best.pt
```

> If generated data already exists in `data/generated/`, §5 will skip preprocessing automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Clone KARMA and external repos

In [ ]:
import subprocess

def sh(cmd):
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise RuntimeError(f'Failed: {" ".join(str(c) for c in cmd)}')

if not ROOT.exists():
    sh(['git', 'clone', '--depth', '1', 'https://github.com/AmTuTi1999/KARMA-.git', str(ROOT)])
else:
    print('KARMA present – pulling'); sh(['git', '-C', str(ROOT), 'pull'])

for name, url in [
    ('WinIT',         'https://github.com/layer6ai-labs/WinIT.git'),
    ('time_interpret','https://github.com/josephenguehard/time_interpret.git'),
    ('TIMING',        'https://github.com/drumpt/TIMING.git'),
]:
    dest = ROOT / name
    if not dest.exists():
        print(f'Cloning {name} ...')
        sh(['git', 'clone', '--depth', '1', url, str(dest)])
    else:
        print(f'{name}: present')

## 3 · Install dependencies

In [ ]:
%%bash
pip install -q "setuptools<81"
pip install -q "numpy<2"
pip install -q \
    pandas scikit-learn scipy \
    torch captum \
    "shap>=0.41,<0.44" timeshap \
    statsmodels joblib tqdm PyYAML \
    "pytorch-lightning>=2.0" "torchmetrics<1.4" \
    reformer-pytorch tigramite
echo Done.

In [ ]:
import torch, pytorch_lightning
print('torch:', torch.__version__,
      '| lightning:', pytorch_lightning.__version__,
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 4 · Link raw data from Drive

In [ ]:
import shutil
RAW.mkdir(parents=True, exist_ok=True)

def link_or_copy(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists() or dst.is_symlink():
        print(f'  {dst.name}: present'); return
    if not src.exists():
        print(f'  WARNING: {src} not on Drive — preprocessing will be skipped'); return
    try:
        dst.symlink_to(src); print(f'  {dst.name}: symlinked')
    except OSError:
        (shutil.copytree if src.is_dir() else shutil.copy2)(str(src), str(dst))
        print(f'  {dst.name}: copied')

# ETT series (single CSV files)
for f in ['ETTh1.csv', 'ETTh2.csv', 'ETTm1.csv', 'ETTm2.csv']:
    link_or_copy(DRIVE_RAW / f, RAW / f)

# Directory-based datasets
link_or_copy(DRIVE_RAW / 'exchange_rate',           RAW / 'exchange_rate')
link_or_copy(DRIVE_RAW / 'beijing_pm2',             RAW / 'beijing_pm2')
link_or_copy(DRIVE_RAW / 'electricityloaddiagrams', RAW / 'electricityloaddiagrams')
link_or_copy(DRIVE_RAW / 'web_traffic',             RAW / 'web_traffic')

## 5 · Preprocess datasets

In [ ]:
import importlib
sys.path.insert(0, str(ROOT))

PREP = [
    # (dataset_name, module, {var: patched_value, ...})
    ('etth1',         'dataset.etth1',         {'CSV_PATH':  str(RAW / 'ETTh1.csv'),       'OUTPUT_DIR': str(GEN / 'etth1')}),
    ('etth2',         'dataset.etth2',         {'CSV_PATH':  str(RAW / 'ETTh2.csv'),       'OUTPUT_DIR': str(GEN / 'etth2')}),
    ('ettm1',         'dataset.ettm1',         {'CSV_PATH':  str(RAW / 'ETTm1.csv'),       'OUTPUT_DIR': str(GEN / 'ettm1')}),
    ('ettm2',         'dataset.ettm2',         {'CSV_PATH':  str(RAW / 'ETTm2.csv'),       'OUTPUT_DIR': str(GEN / 'ettm2')}),
    # exchange_rate: preprocessor uses RAW_DIR (directory) + auto-downloads the .txt.gz
    ('exchange_rate', 'dataset.exchange_rate', {'RAW_DIR':   str(RAW / 'exchange_rate'),   'OUTPUT_DIR': str(GEN / 'exchange_rate')}),
    # beijing: module filename is beijing_pm2_ (trailing underscore); variable is DATA_PATH
    ('beijing_pm25',  'dataset.beijing_pm2_',  {'DATA_PATH': str(RAW / 'beijing_pm2' / 'PRSA_data_2010.1.1-2014.12.31.csv'),
                                                'OUTPUT_DIR': str(GEN / 'beijing_pm25')}),
    ('electricity',   'dataset.electricity',   {'DATA_PATH': str(RAW / 'electricityloaddiagrams' / 'LD2011_2014.txt'),
                                                'OUTPUT_DIR': str(GEN / 'electricity')}),
    ('web_traffic',   'dataset.web_traffic',   {'DATA_PATH': str(RAW / 'web_traffic' / 'kaggle_web_traffic_dataset_without_missing_values.tsf'),
                                                'OUTPUT_DIR': str(GEN / 'web_traffic')}),
]

active = set(DATASETS_RUN)
for ds, mod_name, patches in PREP:
    if ds not in active:
        continue
    if (GEN / ds / 'X_train.npy').exists():
        print(f'{ds}: preprocessed ✓'); continue
    print(f'\n─── Preprocessing {ds} ───')
    mod = importlib.import_module(mod_name)
    for k, v in patches.items():
        setattr(mod, k, v)
        mod.__dict__[k] = v
    mod.main()
    print(f'{ds}: done ✓')

## 6 · Link pre-trained checkpoints from Drive

In [ ]:
for d in [f'{ds}_{arch}' for ds in DATASETS_RUN for arch in ['lstm','tcn']]:
    src = DRIVE_CKPT / d / 'best.pt'
    dst_dir = CKPT_ROOT / d
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / 'best.pt'
    if dst.exists() or dst.is_symlink():
        print(f'  {d}: present')
    elif src.exists():
        dst.symlink_to(src); print(f'  {d}: symlinked')
    else:
        print(f'  {d}: NOT FOUND on Drive — run §7')

## 7 · (Optional) Train from scratch

In [ ]:
for ds in DATASETS_RUN:
    for arch in ['lstm', 'tcn']:
        ckpt = CKPT_ROOT / f'{ds}_{arch}' / 'best.pt'
        if ckpt.exists():
            print(f'{ds}/{arch}: exists'); continue
        print(f'\nTraining {arch.upper()} on {ds} ...')
        subprocess.run(
            [sys.executable, '-m', 'pipeline.training_pipeline',
             '--dataset', ds, '--model', arch, '--epochs', str(TRAIN_EPOCHS)],
            cwd=str(ROOT)
        )

## 8 · Pre-flight check

In [ ]:
import torch
ok = True
for ds in DATASETS_RUN:
    for f in ['X_train.npy', 'X_test.npy']:
        if not (GEN / ds / f).exists():
            print(f'  MISSING {GEN/ds/f} ← §5'); ok = False
    for arch in ['lstm', 'tcn']:
        ckpt = CKPT_ROOT / f'{ds}_{arch}' / 'best.pt'
        if not ckpt.exists():
            print(f'  MISSING {ckpt} ← §6 or §7'); ok = False
        else:
            try:
                torch.load(str(ckpt), map_location='cpu', weights_only=True)
                print(f'  {ds}/{arch}: ok')
            except Exception as e:
                print(f'  CORRUPT {ckpt}: {e}'); ok = False
print('\nAll checks passed ✓' if ok else '\n⚠  Fix issues above')

## 9 · Run 5-fold CV experiment

In [ ]:
import torch
EXP_CKPT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = ('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE == 'auto' else DEVICE

cmd = [
    sys.executable, '-m', 'experiments.comparison_realdata_cv',
    '--datasets', *DATASETS_RUN,
    '--n_folds',         str(N_FOLDS),
    '--device',          device,
    '--lag_only',
    '--model_ckpt_dir',  str(CKPT_ROOT),
    '--ckpt_dir',        str(EXP_CKPT),
    '--out_dir',         str(OUT_DIR),
    '--dynamask_epochs',     '100',
    '--winit_epochs',        '100',
    '--extremalmask_epochs', '300',
]
print('Device:', device)
print('Command:', ' '.join(str(c) for c in cmd))

In [ ]:
with subprocess.Popen(
    cmd, cwd=str(ROOT),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end='', flush=True)

if proc.returncode != 0:
    raise RuntimeError(f'Experiment failed (exit {proc.returncode})')
print('\nDone ✓')

## 10 · Display mean ± std results

In [ ]:
import json, pandas as pd

METHODS = ['fo','dynamask','winit','ig','timeshap','karma',
           'extremalmask','timing']
ARCHS   = ['gru','lstm','tcn']
METRICS = [
    ('lag_auc',    'Lag AUC  ↑'),
    ('lag_drop25', 'Lag Drop@25%  ↑'),
    ('auc',        'Cell AUC  ↑'),
    ('drop25',     'Cell Drop@25%  ↑'),
]

result_files = sorted(OUT_DIR.glob(f'*_cv{N_FOLDS}_results.json'))
all_results  = [json.loads(f.read_text()) for f in result_files]
print(f'Loaded {len(all_results)} file(s):', [f.stem for f in result_files])

rows = []
for r in all_results:
    ds = r.get('dataset', '?')
    for arch in ARCHS:
        for m in METHODS:
            mean = r.get(f'{arch}_{m}_lag_auc_mean')
            if mean is None:
                continue
            row = {'dataset': ds, 'arch': arch, 'method': m}
            for key, label in METRICS:
                mu  = r.get(f'{arch}_{m}_{key}_mean')
                sig = r.get(f'{arch}_{m}_{key}_std')
                row[label] = f'{mu:.4f} ± {sig:.4f}' if mu is not None else '—'
            rows.append(row)

df = pd.DataFrame(rows)
for _, label in METRICS:
    if label not in df.columns:
        continue
    print(f'\n{"═"*100}')
    print(f'  {label}')
    print('═'*100)
    pivot = df.pivot_table(
        index=['dataset','arch'], columns='method',
        values=label, aggfunc='first'
    ).reindex(columns=[m for m in METHODS if m in df.method.values])
    display(pivot)

## 11 · Per-fold breakdown (optional)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Choose which dataset, arch, and metric to plot
PLOT_DS     = 'ettm1'
PLOT_ARCH   = 'lstm'
PLOT_METRIC = 'lag_auc'

r = next((x for x in all_results if x['dataset'] == PLOT_DS), None)
if r is None:
    print(f'{PLOT_DS} not in results'); raise SystemExit

fold_data = {}
for m in METHODS:
    key = f'{PLOT_ARCH}_{m}_{PLOT_METRIC}_folds'
    if key in r and r[key]:
        fold_data[m] = r[key]

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(fold_data))
labels = list(fold_data.keys())

for i, (m, vals) in enumerate(fold_data.items()):
    vals = np.array(vals)
    ax.bar(i, vals.mean(), yerr=vals.std(), capsize=4, alpha=0.8, label=m)

ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel(PLOT_METRIC)
ax.set_title(f'{PLOT_DS} / {PLOT_ARCH} — {PLOT_METRIC}  (mean ± std, {N_FOLDS} folds)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 12 · Save results to Drive

In [ ]:
import shutil
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
for f in result_files:
    shutil.copy2(str(f), str(DRIVE_OUT / f.name))
    print(f'  {f.name} → Drive')

summary = OUT_DIR / f'summary_cv{N_FOLDS}.json'
if summary.exists():
    shutil.copy2(str(summary), str(DRIVE_OUT / summary.name))
    print(f'  {summary.name} → Drive')

# Persist any newly trained checkpoints back to Drive
for d in [f'{ds}_{arch}' for ds in DATASETS_RUN for arch in ['lstm','tcn']]:
    src = CKPT_ROOT / d / 'best.pt'
    if src.exists() and not src.is_symlink():
        dst_dir = DRIVE_CKPT / d
        dst_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(src), str(dst_dir / 'best.pt'))
        print(f'  checkpoint {d} → Drive')

print(f'\nAll outputs at {DRIVE_OUT}')